# CliniqAI — brain_mri Specialist v2 Retrain

Retrains the `brain_mri` specialist from scratch on the Brain_MRI dataset
(`E:\programming\Brain_MRI\data\raw`) — 30 classes: 10 pathology
categories × 3 MRI sequence types (T1, T1C+, T2), kept as 30 distinct
classes per decision. `Normal` is already present as its own category, so
no merge from CliniqAI's `notumor` folder is needed.

**Settings note:** this version prioritizes running reliably to completion
over squeezing out maximum throughput. `num_workers=4` and batch size 64 are
the values already proven stable on this machine for the other specialists
and the gatekeeper — not the theoretical CPU-core maximum. A higher worker
count caused Windows shared-memory exhaustion and hangs in earlier runs.
Every improved epoch is checkpointed to disk, so an interrupted run resumes
instead of restarting from scratch.

In [1]:
# Step 1 — imports, reproducibility, device setup.

import os
import sys
import random
import copy
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

sys.path.append(str(Path(r"E:\programming\CliniqAI").resolve()))
from scripts.brain_mri_v2_dataset import BrainMRIDataset, build_samples, validate_images

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

# Fixed at 4 — this is the value already proven stable elsewhere on this
# project. Higher counts (e.g. CPU_COUNT-2) caused shared-memory exhaustion
# and hangs in Windows' interactive Jupyter kernel. Not worth chasing a
# faster number here; 4 keeps the GPU fed without destabilizing the run.
NUM_WORKERS = 4

print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: CUDA not available — training will run on CPU and be much slower. "
          "If you expected a GPU here, check that this notebook's kernel is the "
          "CliniqAI conda env (Kernel menu / jupyter kernelspec list), not a different Python install.")

Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU, VRAM: 6.4 GB


In [2]:
# Step 2 — inspect the dataset structure (sanity check before building the run).

DATA_ROOT = Path(r"E:\programming\Brain_MRI\data\raw")
assert DATA_ROOT.exists(), f"Path not found: {DATA_ROOT}"

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
subdirs = sorted([d for d in DATA_ROOT.iterdir() if d.is_dir()])
counts = {d.name: sum(1 for f in d.rglob("*") if f.suffix.lower() in IMG_EXT) for d in subdirs}

print(f"{len(counts)} classes, {sum(counts.values())} images total")
for name, n in counts.items():
    print(f"  {name:30s} {n:5d}")

MERGE_NOTUMOR = False  # Normal T1/T1C+/T2 already present as their own classes
USE_WEIGHTED_SAMPLER = True  # imbalance ratio ~8.3x (max 977 / min 118)

30 classes, 11300 images total
  Astrocytoma T1                   396
  Astrocytoma T1C+                 441
  Astrocytoma T2                   281
  Ependymoma T1                    313
  Ependymoma T1C+                  374
  Ependymoma T2                    350
  Glioma T1                        522
  Glioma T1C+                      549
  Glioma T2                        405
  Hemangiopericytoma T1            178
  Hemangiopericytoma T1C+          307
  Hemangiopericytoma T2            118
  Meningioma T1                    636
  Meningioma T1C+                  977
  Meningioma T2                    458
  Neurocytoma T1                   190
  Neurocytoma T1C+                 254
  Neurocytoma T2                   174
  Normal T1                        415
  Normal T1C+                      264
  Normal T2                        379
  Oligodendroglioma T1             221
  Oligodendroglioma T1C+           192
  Oligodendroglioma T2             137
  Other T1                       

In [3]:
# Step 3 — build the sample list + class names directly from folder names
# (never manually re-typed — avoids the label-order bugs already hit twice
# on this project with kidney_ct and pneumonia).

samples, class_names, class_to_idx = build_samples(DATA_ROOT)
print(f"{len(samples)} samples, {len(class_names)} classes:")
for name, idx in class_to_idx.items():
    print(f"  {idx}: {name}")

11300 samples, 30 classes:
  0: Astrocytoma T1
  1: Astrocytoma T1C+
  2: Astrocytoma T2
  3: Ependymoma T1
  4: Ependymoma T1C+
  5: Ependymoma T2
  6: Glioma T1
  7: Glioma T1C+
  8: Glioma T2
  9: Hemangiopericytoma T1
  10: Hemangiopericytoma T1C+
  11: Hemangiopericytoma T2
  12: Meningioma T1
  13: Meningioma T1C+
  14: Meningioma T2
  15: Neurocytoma T1
  16: Neurocytoma T1C+
  17: Neurocytoma T2
  18: Normal T1
  19: Normal T1C+
  20: Normal T2
  21: Oligodendroglioma T1
  22: Oligodendroglioma T1C+
  23: Oligodendroglioma T2
  24: Other T1
  25: Other T1C+
  26: Other T2
  27: Schwannoma T1
  28: Schwannoma T1C+
  29: Schwannoma T2


In [4]:
# Step 4 — pre-validation pass: drop corrupted/unreadable files up front.

samples = validate_images(samples)
print(f"{len(samples)} samples remain after validation.")

[validate_images] 11300 OK, 0 corrupted/unreadable
11300 samples remain after validation.


In [5]:
# Step 5 — stratified train/val/test split (80/10/10) + class weights for
# the imbalance (ratio ~8.3x).

labels = [label for _, label in samples]

train_samples, temp_samples, train_labels, temp_labels = train_test_split(
    samples, labels, test_size=0.20, stratify=labels, random_state=SEED
)
val_samples, test_samples, val_labels, test_labels = train_test_split(
    temp_samples, temp_labels, test_size=0.50, stratify=temp_labels, random_state=SEED
)

print(f"Train: {len(train_samples)}  Val: {len(val_samples)}  Test: {len(test_samples)}")

class_counts = np.bincount(train_labels, minlength=len(class_names))
for name, c in zip(class_names, class_counts):
    print(f"  {name:30s} {c:5d}")

train_sampler = None
if USE_WEIGHTED_SAMPLER:
    class_weights = 1.0 / np.maximum(class_counts, 1)
    sample_weights = [class_weights[label] for label in train_labels]
    train_sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_samples), replacement=True)
    print("\nUsing WeightedRandomSampler.")

loss_class_weights = torch.tensor(1.0 / np.maximum(class_counts, 1), dtype=torch.float32)
loss_class_weights = (loss_class_weights / loss_class_weights.sum() * len(class_names)).to(DEVICE)

Train: 9040  Val: 1130  Test: 1130
  Astrocytoma T1                   317
  Astrocytoma T1C+                 353
  Astrocytoma T2                   225
  Ependymoma T1                    250
  Ependymoma T1C+                  299
  Ependymoma T2                    280
  Glioma T1                        418
  Glioma T1C+                      439
  Glioma T2                        324
  Hemangiopericytoma T1            142
  Hemangiopericytoma T1C+          246
  Hemangiopericytoma T2             94
  Meningioma T1                    509
  Meningioma T1C+                  782
  Meningioma T2                    366
  Neurocytoma T1                   152
  Neurocytoma T1C+                 203
  Neurocytoma T2                   139
  Normal T1                        332
  Normal T1C+                      211
  Normal T2                        303
  Oligodendroglioma T1             177
  Oligodendroglioma T1C+           154
  Oligodendroglioma T2             110
  Other T1                   

In [6]:
# Step 6 — transforms. 224x224, ImageNet normalization. Mild augmentation
# on train only.

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [7]:
# Step 7 — Datasets + DataLoaders. Batch 64 + num_workers=4 is the
# proven-stable combination on this machine — not the theoretical ceiling,
# but the value that actually finishes runs without crashing.

BATCH_SIZE = 64

train_ds = BrainMRIDataset(train_samples, transform=train_transform)
val_ds = BrainMRIDataset(val_samples, transform=eval_transform)
test_ds = BrainMRIDataset(test_samples, transform=eval_transform)

loader_kwargs = dict(num_workers=NUM_WORKERS, pin_memory=True,
                      persistent_workers=True, prefetch_factor=4)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=train_sampler,
                           shuffle=(train_sampler is None), **loader_kwargs)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

print(f"Batches — train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}")

Batches — train: 142, val: 18, test: 18


In [8]:
# Step 8 — model: EfficientNet-B0, wide head (hidden=512), matching the
# other wide-head specialists so model_loader.py picks it up unchanged.

def build_model(num_classes):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2),
        nn.Linear(in_features, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.2),
        nn.Linear(512, num_classes),
    )
    return model

model = build_model(len(class_names)).to(DEVICE)
model = model.to(memory_format=torch.channels_last)
print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=512, bias=True)
  (2): ReLU(inplace=True)
  (3): Dropout(p=0.2, inplace=False)
  (4): Linear(in_features=512, out_features=30, bias=True)
)


In [9]:
# Step 9 — AMP, optimizer, scheduler.
# torch.amp.autocast (not the deprecated torch.cuda.amp.autocast) — same
# effect: roughly halves activation memory, which is what makes batch 64
# safe on 6GB VRAM.

criterion = nn.CrossEntropyLoss(weight=loss_class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
scaler = torch.amp.GradScaler(device=DEVICE.type, enabled=(DEVICE.type == "cuda"))

EARLY_STOP_PATIENCE = 8
MAX_EPOCHS = 60

INPROGRESS_CKPT = Path(r"E:\programming\CliniqAI\models\specialists\brain_tumor_specialist_v2.checkpoint_inprogress.pt")

In [10]:
# Step 10 — resume from a prior checkpoint if one exists. Skip this cell
# (or don't run it) for a genuinely fresh run; run it to pick back up after
# an interruption instead of retraining from epoch 1.

start_epoch = 1
best_val_loss = float("inf")
history = []

if INPROGRESS_CKPT.exists():
    print(f"Found existing checkpoint: {INPROGRESS_CKPT}")
    ckpt = torch.load(INPROGRESS_CKPT, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    start_epoch = ckpt.get("epoch", 0) + 1
    best_val_loss = ckpt.get("val_loss", float("inf"))
    print(f"Resuming from epoch {start_epoch}, best val_loss so far: {best_val_loss:.4f}")
else:
    print("No existing checkpoint found — starting fresh from epoch 1.")

No existing checkpoint found — starting fresh from epoch 1.


In [11]:
# Step 11 — training loop. Checkpoints to disk on every improved epoch, so
# an interruption (crash, hang, closed laptop lid) never costs more than the
# epochs since the last improvement.

def run_epoch(loader, train_mode):
    model.train(train_mode)
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        labels = labels.to(DEVICE, non_blocking=True)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train_mode):
            with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                outputs = model(images)
                loss = criterion(outputs, labels)

            if train_mode:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += images.size(0)

    return total_loss / total, correct / total


best_state = None
epochs_without_improvement = 0

for epoch in range(start_epoch, MAX_EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, train_mode=True)
    val_loss, val_acc = run_epoch(val_loader, train_mode=False)
    scheduler.step(val_loss)

    history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                     "val_loss": val_loss, "val_acc": val_acc})
    print(f"Epoch {epoch:3d} | train loss {train_loss:.4f} acc {train_acc:.4f} "
          f"| val loss {val_loss:.4f} acc {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
        torch.save({
            "model_state_dict": best_state,
            "class_names": class_names,
            "classes": class_names,
            "epoch": epoch,
            "val_loss": val_loss,
            "val_acc": val_acc,
        }, INPROGRESS_CKPT)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

if best_state is not None:
    model.load_state_dict(best_state)
print(f"\nBest val loss: {best_val_loss:.4f}")

Epoch   1 | train loss 1.4031 acc 0.4851 | val loss 1.1670 acc 0.5469
Epoch   2 | train loss 0.6033 acc 0.7416 | val loss 1.1059 acc 0.5655
Epoch   3 | train loss 0.4195 acc 0.8253 | val loss 0.6013 acc 0.7841
Epoch   4 | train loss 0.2756 acc 0.8770 | val loss 0.6848 acc 0.7858
Epoch   5 | train loss 0.2883 acc 0.8821 | val loss 0.5563 acc 0.8381
Epoch   6 | train loss 0.2005 acc 0.9111 | val loss 0.5103 acc 0.8611
Epoch   7 | train loss 0.1425 acc 0.9371 | val loss 0.4329 acc 0.8788
Epoch   8 | train loss 0.1763 acc 0.9254 | val loss 0.4921 acc 0.8416
Epoch   9 | train loss 0.1866 acc 0.9216 | val loss 0.4665 acc 0.8708
Epoch  10 | train loss 0.1727 acc 0.9299 | val loss 0.4209 acc 0.8858
Epoch  11 | train loss 0.1229 acc 0.9479 | val loss 0.3223 acc 0.9159
Epoch  12 | train loss 0.0877 acc 0.9619 | val loss 0.3700 acc 0.9080
Epoch  13 | train loss 0.1262 acc 0.9455 | val loss 0.5008 acc 0.8770
Epoch  14 | train loss 0.1374 acc 0.9421 | val loss 0.3521 acc 0.9080
Epoch  15 | train lo

In [12]:
# Step 12 — final evaluation on the held-out test set: overall accuracy +
# per-class breakdown.

if "model" not in dir():
    raise RuntimeError("model is undefined — run Steps 8-11 first.")

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            outputs = model(images)
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))
print("Confusion matrix (rows=true, cols=pred):")
print(confusion_matrix(all_labels, all_preds))

                         precision    recall  f1-score   support

         Astrocytoma T1     1.0000    0.9744    0.9870        39
       Astrocytoma T1C+     1.0000    0.9545    0.9767        44
         Astrocytoma T2     0.9286    0.9286    0.9286        28
          Ependymoma T1     0.9375    0.9677    0.9524        31
        Ependymoma T1C+     0.9744    1.0000    0.9870        38
          Ependymoma T2     1.0000    0.9429    0.9706        35
              Glioma T1     1.0000    0.9423    0.9703        52
            Glioma T1C+     1.0000    1.0000    1.0000        55
              Glioma T2     0.9250    0.9024    0.9136        41
  Hemangiopericytoma T1     1.0000    1.0000    1.0000        18
Hemangiopericytoma T1C+     1.0000    0.9355    0.9667        31
  Hemangiopericytoma T2     1.0000    1.0000    1.0000        12
          Meningioma T1     0.9688    0.9841    0.9764        63
        Meningioma T1C+     1.0000    0.9898    0.9949        98
          Meningioma T2 

In [13]:
# Step 13 — save the final checkpoint in CliniqAI's expected format.

OUT_PATH = Path(r"E:\programming\CliniqAI\models\specialists\brain_tumor_specialist_v2.pt")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

checkpoint = {
    "model_state_dict": model.state_dict(),
    "class_names": class_names,
    "classes": class_names,
    "num_classes": len(class_names),
    "architecture": "efficientnet_b0_wide512",
}
torch.save(checkpoint, OUT_PATH)
print(f"Saved to {OUT_PATH}")

with open(OUT_PATH.with_suffix(".history.json"), "w") as f:
    json.dump(history, f, indent=2)

# Clean up the in-progress checkpoint now that the final one is saved.
if INPROGRESS_CKPT.exists():
    INPROGRESS_CKPT.unlink()
    print(f"Removed {INPROGRESS_CKPT}")

print("""
Next steps (not done by this notebook):
1. Copy checkpoint into backend/models/specialists/ (manual sync, no build step yet).
2. Point config.py's brain_mri specialist entry at brain_tumor_specialist_v2.pt.
3. Confirm model_loader.py picks up the new class list via the classes key.
4. Re-run the acoustic neuroma / OOD sanity test plus a fresh manual_sanity_check pass.
5. Upload the new checkpoint to the shared Google Drive.
""")

Saved to E:\programming\CliniqAI\models\specialists\brain_tumor_specialist_v2.pt
Removed E:\programming\CliniqAI\models\specialists\brain_tumor_specialist_v2.checkpoint_inprogress.pt

Next steps (not done by this notebook):
1. Copy checkpoint into backend/models/specialists/ (manual sync, no build step yet).
2. Point config.py's brain_mri specialist entry at brain_tumor_specialist_v2.pt.
3. Confirm model_loader.py picks up the new class list via the classes key.
4. Re-run the acoustic neuroma / OOD sanity test plus a fresh manual_sanity_check pass.
5. Upload the new checkpoint to the shared Google Drive.

